Template to generate farmer queries - not needed for now as we are using the questions that are manually generated.

In [76]:
import json
import random
from openai import OpenAI
import os
from dotenv import load_dotenv
import csv

In [77]:
os.chdir(r"D:\Kenpath\oan-evaluation")

In [78]:
load_dotenv()

True

In [79]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
MODEL_NAME = "gpt-5" 
OUTPUT_FILE = "farmer_queries.csv"
TOTAL_QUERIES = 25

In [80]:
client = OpenAI(api_key=OPENAI_API_KEY)


In [81]:
categories = [
    "MahaDBT",
    "Warehouse",
    "CHC",
    "KVK",
    "Soil Labs",
    "Advisory",
    "Weather Historical",
    "Weather Forecast",
    "Agri Assistant Contact",
    "Mandi",
    "Scheme"
]

intents = [
    "status_check",
    "information",
    "advisory",
    "contact",
    "historical weather",
    "forecast",
    "mandi price",
    "irrigation planning",
    "application guidance",
    "storage",
    "location"
]
agri = ["Yes", "No"]

In [ ]:
def build_prompt(category, agristack, intent, examples=None):
    # Default examples if none provided
    if examples is None:
        examples = [
            "बँक खाते पडताळणी अयशस्वी झाल्यास महाडीबीटीमध्ये दुरुस्ती कशी करावी?",
            "तांदूळ दीर्घकाळ साठवण्यासाठी गोदामात कोणती खबरदारी घेतली जाते?",
            "गाव पातळीवर उपलब्ध असलेल्या CHC मध्ये ट्रॅक्टर भाड्याचे प्रति तास दर काय आहेत?",
            "KVK मध्ये पीक रोगनिदानासाठी नमुना कोठे जमा करावा लागतो?",
            "विशेष सूक्ष्म अन्नद्रव्य माती चाचणीसाठी वेगळे शासकीय शुल्क लागते का?"
        ]
    
    # Format examples for the prompt
    examples_text = "\n".join(f"- {ex}" for ex in examples)
    
    return f"""
You are an agricultural AI generating SHORT, DIRECT farmer questions from Maharashtra in Marathi.

Task:
Generate ONE SHORT, FOCUSED farmer question that matches the style and length of the examples.

Reference Examples:
{examples_text}

Context (DO NOT MENTION IN QUESTIONS):
- Category: {category}
- Intent: {intent}
- Agristack Data Available: {agristack}

CRITICAL ASSUMPTION RULES (MANDATORY):

1. For the following categories, the question MUST ALWAYS assume
   that farmer-specific data is AVAILABLE (implicit identification).
   ❗ Do NOT generate generic or anonymous questions for these.

   ALWAYS-AGRISTACK AVAILABLE categories:
   - MahaDBT
   - Weather Historical
   - Weather Forecast
   - Mandi
   - Soil Labs
   - Warehouse
   - CHC
   - KVK
   - Agri Assistant Contact

   These questions should sound like:
   - status check, booking, availability, or farmer-linked service

2. For ALL OTHER categories (e.g., Advisory, Scheme, General Agriculture):
   - Generate BOTH types of questions across runs:
     a) Questions assuming farmer data IS available
     b) Questions assuming farmer data is NOT available (generic inquiry)
   - Follow the provided agristack value to decide which type to generate.

STRICT OUTPUT RULES:
- Generate ONLY ONE question
- Length: 10–15 words MAX
- Match the style and tone of reference examples
- NEVER mention:
  "Agristack", "database", "system", "platform", IDs, or technical terms
- The question must sound like a real farmer speaking naturally
- Focus on ONE clear problem or need
- Direct, practical, no explanation-
- Must cover questions with all categories including agristack yes and no
- Language: Marathi only (Devanagari preferred)

Category Focus Guidance:
- MahaDBT: application status, correction, eligibility, benefits
- Weather: current conditions, 7-day historical and forecast.
- Mandi: today’s prices, selling decision, market availability
- KVK / CHC / Soil Lab: location, contact, sample submission
- Warehouse: storage safety, charges, booking
- Advisory: specific pest, disease, or crop action

Generate ONLY the question. No extra text.
"""


In [83]:
def generate_farmer_queries(num_queries=1, save_to_file=None, show_progress=True):
    """
    Generate farmer queries with AI.
    
    Args:
        num_queries (int): Number of queries to generate (default: 1)
        save_to_file (str): Optional filename to save results (default: None)
        show_progress (bool): Whether to show progress messages (default: True)
    
    Returns:
        list: List of generated query dictionaries
    """
    dataset = []
    
    for i in range(num_queries):
        # Select random parameters
        category = random.choice(categories)
        intent = random.choice(intents)
        agristack = random.choice(agri)
        
        # Build prompt and call AI
        prompt = build_prompt(category, agristack, intent)
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": "Generate realistic farmer questions in Marathi with Maharashtra agricultural context."},
                {"role": "user", "content": prompt}
            ],
            temperature=1.0
        )
        
        # Parse response - content is plain text, not JSON
        content = response.choices[0].message.content.strip()
        
        # Create query record
        query_record = {
            "user_query": content,  # Use content directly as the question
            "category": category,
            "agristack": agristack
        }
        
        dataset.append(query_record)
        
        if show_progress:
            print(f"Generated {i + 1}/{num_queries}: {content}")
    
    # Save to file if requested
    if save_to_file:
        # Save as CSV
        with open(save_to_file, "w", newline="", encoding="utf-8") as f:
            if dataset:
                writer = csv.DictWriter(f, fieldnames=dataset[0].keys())
                writer.writeheader()
                writer.writerows(dataset) 
    return dataset

In [84]:
if __name__ == "__main__":
    print("Generating farmer queries")
    data = generate_farmer_queries(TOTAL_QUERIES, save_to_file=OUTPUT_FILE)
    print(f"Done!")

Generating farmer queries
Generated 1/25: माझ्या शेतासाठी ठिबक सिंचन अनुदान मंजुरीची आताची स्थिती काय आहे?
Generated 2/25: माझ्या गावासाठी संबंधित कृषी सहाय्यक कार्यालयाचा पत्ता आणि नकाशा पाठवू शकाल का?
Generated 3/25: माझ्या गावासाठी जवळची सरकारी मृदा परीक्षण प्रयोगशाळा नमुना जमा करण्यासाठी कुठे आहे?
Generated 4/25: माझ्या जमिनीपासून २० किमी आत जवळचे नोंदणीकृत गोदाम कुठे आहे?
Generated 5/25: महाडीबीटीमध्ये माझ्या अनुदानाची अंदाजे खात्यात जमा तारीख आता काय आहे?
Generated 6/25: माझ्या गावाजवळील शासकीय मृदा परीक्षण प्रयोगशाळेचा फोन नंबर आणि पत्ता द्या.
Generated 7/25: माझ्या ठिबक सिंचन अनुदानाच्या महाडीबीटी अर्जाची स्थिती आणि पुढची पायरी काय?
Generated 8/25: KVK मध्ये माझ्या धान्य साठवण सल्ल्यासाठी स्लॉट आज उपलब्ध आहे का?
Generated 9/25: माझ्या शेतातील कांद्यासाठी पुढील ७ दिवसांचे सिंचन वेळापत्रक काय ठेवावे?
Generated 10/25: माझ्या गावातील CHC समन्वयकाचा संपर्क क्रमांक मला आत्ता पाठवाल का?
Generated 11/25: माझ्या शेताच्या ठिकाणाचा गेल्या पाच वर्षांचा मासिक पावसाचा सारांश मिळेल का?
Genera